# Double Heston PINN Baseline V1 — Kaggle P100
This notebook runs the frozen synthetic fixed-parameter forward-pricing baseline. Supply the repository URL and exact branch or commit; do not use test metrics for tuning.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'torch==2.7.1', '--index-url', 'https://download.pytorch.org/whl/cu126'], check=True)

The pinned CUDA 12.6 wheel retains Pascal `sm_60` support required by Kaggle P100. Because torch is not imported before installation, a fresh Kaggle kernel can verify it immediately.

In [ ]:
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
import torch
print('torch', torch.__version__, 'cuda build', torch.version.cuda)
assert torch.__version__.startswith('2.7.1'), torch.__version__
assert torch.version.cuda == '12.6', torch.version.cuda
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator first.'
assert torch.cuda.get_device_capability(0) == (6, 0), torch.cuda.get_device_capability(0)
assert 'sm_60' in torch.cuda.get_arch_list(), torch.cuda.get_arch_list()
print('GPU:', torch.cuda.get_device_name(0), 'capability:', torch.cuda.get_device_capability(0))
print('compiled architectures:', torch.cuda.get_arch_list())
print('CUBLAS_WORKSPACE_CONFIG:', os.environ['CUBLAS_WORKSPACE_CONFIG'])

In [ ]:
REPO_URL = os.environ.get('MENTOR_DH_PINN_REPO_URL', '')
TARGET_REF = os.environ.get('MENTOR_DH_PINN_TARGET_REF', '')  # exact branch or commit supplied by operator
assert REPO_URL and TARGET_REF, 'Set MENTOR_DH_PINN_REPO_URL and MENTOR_DH_PINN_TARGET_REF.'
subprocess.run(['git', 'clone', '--no-checkout', REPO_URL, '/kaggle/working/dh_mentor_pinn_basics'], check=True)
os.chdir('/kaggle/working/dh_mentor_pinn_basics')
subprocess.run(['git', 'fetch', 'origin', TARGET_REF], check=True)
subprocess.run(['git', 'checkout', '--detach', 'FETCH_HEAD'], check=True)
print('checked out', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
import hashlib, pathlib
dataset = pathlib.Path('data/final_r2_clean_10000/surfaces.jsonl')
actual = hashlib.sha256(dataset.read_bytes()).hexdigest()
expected = '148b579a4f6ce572e34796e872479c4c016c89bbcd20438c2bb62d6b6960f1f6'
assert actual == expected, (actual, expected)
print('dataset SHA verified:', actual)

In [ ]:
%pip install -q numpy scipy PyYAML matplotlib

In [ ]:
subprocess.run([sys.executable, 'scripts/mentor_dh_pinn/run_baseline.py', '--device', 'cuda'], check=True)
subprocess.run([sys.executable, 'scripts/mentor_dh_pinn/evaluate_baseline.py'], check=True)
subprocess.run([sys.executable, 'scripts/mentor_dh_pinn/make_figures.py'], check=True)

In [ ]:
import json
metrics = json.loads(pathlib.Path('outputs/mentor_dh_pinn_baseline_v1/test_metrics.json').read_text())
metrics

In [ ]:
from IPython.display import display, Image
for figure in sorted(pathlib.Path('outputs/mentor_dh_pinn_baseline_v1/figures').glob('*.png')):
    print(figure.name)
    display(Image(filename=str(figure)))

In [ ]:
import shutil
bundle = shutil.make_archive('/kaggle/working/mentor_dh_pinn_baseline_v1_bundle', 'zip', 'outputs/mentor_dh_pinn_baseline_v1')
print(bundle)